# Chạy LLM extractor (Qwen2.5-7B) trên Google Colab
Self-host ≤9B, KHÔNG API ngoài. Runtime → Change runtime type → **GPU** (T4/L4/A100).
Mẹo: cache model vào Google Drive để khỏi tải lại 15GB mỗi phiên.

In [1]:
# 1) (Khuyến nghị) Mount Drive + cache HuggingFace vào Drive để khỏi tải lại model
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
    print('HF_HOME =', os.environ['HF_HOME'])
except Exception as e:
    print('Bỏ qua Drive:', e)

Mounted at /content/drive
HF_HOME = /content/drive/MyDrive/hf_cache


In [2]:
# 2) Lấy code
%cd /content
![ -d viettel-medreason ] || git clone https://github.com/tienndat1904/viettel-medreason.git
%cd /content/viettel-medreason
!git pull -q

/content
Cloning into 'viettel-medreason'...
remote: Enumerating objects: 552, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 552 (delta 1), reused 2 (delta 0), pack-reused 534 (from 1)
Receiving objects: 100% (552/552), 3.84 MiB | 8.01 MiB/s, done.
Resolving deltas: 100% (220/220), done.
/content/viettel-medreason


In [3]:
# 3) Cài dependencies — KHÔNG cài lại transformers/torch (dùng bản sẵn của Colab, đã hợp triton 3.x).
#    Chỉ cài core nhẹ + CẬP NHẬT bitsandbytes (fix 'triton.ops' + binary CUDA 12.8) + peft/accelerate.
!pip install -q -r requirements.txt
!pip install -q -U bitsandbytes peft accelerate
import torch, transformers, bitsandbytes as bnb
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| bitsandbytes', bnb.__version__, '| CUDA', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 35.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas

In [4]:
# 4) SMOKE TEST 3 file trước (kiểm model + JSON parse; lần đầu tải model ~15GB)
import os, shutil
os.makedirs('smoke/input', exist_ok=True)
for n in [1, 5, 50]:
    shutil.copy(f'data/test/input/{n}.txt', f'smoke/input/{n}.txt')
!python src/pipeline.py --input smoke/input --output smoke/out --backend llm
print('----- smoke/out/5.json -----')
!cat smoke/out/5.json

[linker] backend=lexical icd_mode=lexical | icd=kb(98186) rxnorm=kb(47719) | synonyms=53 brands=4150
[pipeline] backend=llm | 3 file | out=smoke/out
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1% 2/339 [00:40<1:52:52, 20.10s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 339/339 [04:59<00:00,  1.13it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[pipeline] xong.
----- s

In [5]:
# 5) Chạy full 100 file (~1–3h; đổi extract.llm_model sang Qwen2.5-3B-Instruct để lặp nhanh)
!python src/pipeline.py --input data/test/input --output output --backend llm

[linker] backend=lexical icd_mode=lexical | icd=kb(98186) rxnorm=kb(47719) | synonyms=53 brands=4150
[pipeline] backend=llm | 100 file | out=output
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1% 2/339 [00:21<1:07:13, 11.97s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 339/339 [02:08<00:00,  2.64it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[pipeline] xong.


In [6]:
# 6) Validate + đóng gói + tải về
!python scripts/package_submission.py --output output --input data/test/input --n 100
from google.colab import files; files.download('output.zip')

✅ Đã tạo output.zip (100 file hợp lệ).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
# 7) Đo trên dev gold (copy output 2 cell này gửi lại để tinh chỉnh)
!python src/eval/scorer.py --pred output --gold data/dev/gold --mode overlap
!python src/eval/eval_linking.py


=== SPAN+TYPE (mode=overlap) ===
P=0.587  R=0.434  F1=0.499   (TP=331 nPred=564 nGold=763)

=== theo type (F1) ===
  CHẨN_ĐOÁN            F1=0.432  (tp=59 pred=63 gold=210)
  KẾT_QUẢ_XÉT_NGHIỆM   F1=0.519  (tp=47 pred=107 gold=74)
  THUỐC                F1=0.718  (tp=51 pred=65 gold=77)
  TRIỆU_CHỨNG          F1=0.427  (tp=108 pred=256 gold=250)
  TÊN_XÉT_NGHIỆM       F1=0.587  (tp=66 pred=73 gold=152)

Assertion exact-set acc = 0.917 (200/218)
Candidate hit (gold∈pred) = 0.318 (35/110)
[linker] backend=lexical icd_mode=lexical | icd=kb(98186) rxnorm=kb(47719) | synonyms=53 brands=4150

=== ICD-10 (CHẨN_ĐOÁN) — chấm phân cấp ===
  mention: 210 | có mã gold: 194 | linker trả ≥1 mã: 181 (coverage=0.86)
  [exact] hit@k=101/194=0.521  top1=0.448   (trùng mã đầy đủ)
  [cat4 ] hit@k=105/194=0.541  top1=0.510   (4 ký tự)
  [cat3 ] hit@k=129/194=0.665  top1=0.655   (nhóm 3 ký tự)

=== RxNorm (THUỐC) — chấm theo HOẠT CHẤT ===
  mention: 77 | có mã gold: 71 | linker trả ≥1 mã: 72 (coverage=0.94